<a href="https://colab.research.google.com/github/JorgeAccardi/auscultacion-presa/blob/main/Aus_An%C3%A1lisis_Exploratorio_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import io
import base64
from datetime import datetime
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

# --- Configuración y estilos ---
instrumentos = [
    "puntos_fijos_mi",
    "puntos_fijos_md",
    "inclinometros",
    "asentamiento",
    "piezometros_electricos",
    "piezometros_casagrande",
    "freatimetros",
    "extensometro"
]
datos_csv = {inst: pd.DataFrame() for inst in instrumentos}
datos_xlsx = {inst: pd.DataFrame() for inst in instrumentos}

def detectar_instrumento(nombre):
    nombre = nombre.lower()
    if "puntosfijos" in nombre or "pf" in nombre:
        if "mi" in nombre:
            return "puntos_fijos_mi"
        elif "md" in nombre:
            return "puntos_fijos_md"
        else:
            return "puntos_fijos_mi"
    elif "incli" in nombre:
        return "inclinometros"
    elif "as" in nombre:
        return "asentamiento"
    elif "pe" in nombre:
        return "piezometros_electricos"
    elif "pcg" in nombre:
        return "piezometros_casagrande"
    elif "frea" in nombre:
        return "freatimetros"
    elif "ext" in nombre:
        return "extensometro"
    return None

# Widgets globales para mantener la UI estable
upload_widget = widgets.FileUpload(
    accept='.csv,.xlsx',
    multiple=True,
    description='Subir archivos',
    style={'button_color': 'lightblue'},
    layout=widgets.Layout(width="350px")
)
output_carga = widgets.Output()
output_tabla = widgets.Output()

instrumento_selector = widgets.Dropdown(
    options=instrumentos,
    description='Instrumento:',
    layout=widgets.Layout(width='260px')
)
origen_selector = widgets.Dropdown(
    options=['csv', 'xlsx'],
    description='Origen:',
    layout=widgets.Layout(width='160px')
)
boton_ver = widgets.Button(
    description='👁️ Ver',
    button_style='success',
    icon='eye',
    layout=widgets.Layout(width='100px')
)
boton_descargar = widgets.Button(
    description='💾 Descargar',
    button_style='info',
    icon='download',
    layout=widgets.Layout(width='130px')
)

# --- Función de carga y progreso ---
def cargar_archivos(change):
    with output_carga:
        clear_output(wait=True)
        archivos = upload_widget.value
        if not archivos:
            display(HTML("<div style='color:#b71c1c;font-weight:bold;'>⚠️ No se subió ningún archivo.</div>"))
            return

        barra_progreso = widgets.FloatProgress(
            value=0, min=0, max=100, description='Progreso:', bar_style='info',
            layout=widgets.Layout(width='80%')
        )
        etiqueta_progreso = widgets.Label(value="0% completado")
        display(barra_progreso, etiqueta_progreso)

        total = len(archivos)
        archivos_exitosos = 0

        for i, (nombre_archivo, archivo_info) in enumerate(archivos.items(), start=1):
            try:
                extension = nombre_archivo.split('.')[-1].lower()
                instrumento = detectar_instrumento(nombre_archivo)
                contenido = archivo_info['content']

                if not instrumento:
                    display(HTML(f"<span style='color:#b71c1c;'>❌ Instrumento no reconocido en archivo: {nombre_archivo}</span>"))
                    continue

                if extension == 'csv':
                    try:
                        try:
                            df = pd.read_csv(io.BytesIO(contenido), encoding='utf-8')
                        except UnicodeDecodeError:
                            df = pd.read_csv(io.BytesIO(contenido), encoding='latin-1')
                        if not df.empty:
                            datos_csv[instrumento] = pd.concat([datos_csv[instrumento], df], ignore_index=True)
                            archivos_exitosos += 1
                            display(HTML(f"<span style='color:#388e3c;'>✔️ {nombre_archivo} cargado como CSV ({len(df)} filas)</span>"))
                        else:
                            display(HTML(f"<span style='color:#ffa000;'>⚠️ CSV vacío: {nombre_archivo}</span>"))
                    except Exception as e:
                        display(HTML(f"<span style='color:#b71c1c;'>❌ Error leyendo CSV {nombre_archivo}: {str(e)}</span>"))
                elif extension == 'xlsx':
                    try:
                        df = pd.read_excel(io.BytesIO(contenido))
                        if not df.empty:
                            datos_xlsx[instrumento] = pd.concat([datos_xlsx[instrumento], df], ignore_index=True)
                            archivos_exitosos += 1
                            display(HTML(f"<span style='color:#388e3c;'>✔️ {nombre_archivo} cargado como XLSX ({len(df)} filas)</span>"))
                        else:
                            display(HTML(f"<span style='color:#ffa000;'>⚠️ XLSX vacío: {nombre_archivo}</span>"))
                    except Exception as e:
                        display(HTML(f"<span style='color:#b71c1c;'>❌ Error leyendo XLSX {nombre_archivo}: {str(e)}</span>"))
                else:
                    display(HTML(f"<span style='color:#b71c1c;'>⚠️ Extensión no soportada: {nombre_archivo}</span>"))
            except Exception as e:
                display(HTML(f"<span style='color:#b71c1c;'>❌ Error general en {nombre_archivo}: {str(e)}</span>"))
            porcentaje = (i / total) * 100
            barra_progreso.value = porcentaje
            etiqueta_progreso.value = f"{porcentaje:.0f}% completado"

        barra_progreso.bar_style = 'success'
        etiqueta_progreso.value = f"✅ {archivos_exitosos}/{total} archivos procesados exitosamente"
        resumen = "<ul>"
        for inst in instrumentos:
            csv_count = len(datos_csv[inst])
            xlsx_count = len(datos_xlsx[inst])
            if csv_count > 0 or xlsx_count > 0:
                resumen += f"<li><b>{inst}</b>: {csv_count} filas CSV, {xlsx_count} filas XLSX</li>"
        resumen += "</ul>"
        display(HTML(f"<div style='margin-top:10px;'><b>Resumen de datos:</b>{resumen}</div>"))

# --- Función para mostrar tabla o mensaje ---
def ver_datos(b):
    with output_tabla:
        clear_output(wait=True)
        instrumento = instrumento_selector.value
        origen = origen_selector.value
        df = datos_csv[instrumento] if origen == 'csv' else datos_xlsx[instrumento]
        if df.empty:
            display(HTML("<div style='color:#b71c1c;font-weight:bold;'>⚠️ No hay datos disponibles para el instrumento y origen seleccionados.</div>"))
        else:
            display(HTML(f"<div style='margin-bottom:10px;'><b>{instrumento.replace('_',' ').title()} ({origen.upper()})</b> - <span style='color:#388e3c'>Filas: {len(df)} | Columnas: {len(df.columns)}</span></div>"))
            display(df.head(5))  # AJUSTE: solo 5 registros

# --- Función para descargar con enlace bonito ---
def descargar_datos(b):
    with output_tabla:
        clear_output(wait=True)
        instrumento = instrumento_selector.value
        origen = origen_selector.value
        df = datos_csv[instrumento] if origen == 'csv' else datos_xlsx[instrumento]
        if df.empty:
            display(HTML("<div style='color:#b71c1c;font-weight:bold;'>⚠️ No hay datos para descargar.</div>"))
            return
        fecha_actual = datetime.now().strftime("%Y%m%d_%H%M%S")
        extension = 'csv' if origen == 'csv' else 'xlsx'
        nombre_archivo = f"{instrumento}_{origen}_{fecha_actual}.{extension}"
        buffer = io.BytesIO()
        try:
            if extension == 'csv':
                df.to_csv(buffer, index=False, encoding='utf-8')
                mime = "text/csv"
            else:
                # openpyxl es necesario para xlsx, pero en Colab/Jupyter viene instalado
                df.to_excel(buffer, index=False, engine='openpyxl')
                mime = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
            buffer.seek(0)
            b64 = base64.b64encode(buffer.read()).decode()
            # NOTA: El enlace debe estar en una sola línea sin saltos para funcionar bien
            html = f"""
            <div style='padding:12px 18px;background:#e3f2fd;border:1.5px solid #2196f3;border-radius:7px;max-width:420px;'>
                <b>Descarga lista:</b><br>
                <span style='color:#1565c0'><b>{nombre_archivo}</b></span><br>
                <a download="{nombre_archivo}" href="data:{mime};base64,{b64}" target="_blank" style="display:inline-block;margin-top:10px;padding:10px 18px;background:#2196f3;color:white;text-decoration:none;border-radius:4px;font-weight:bold;">
                   📥 Descargar archivo
                </a>
            </div>
            """
            display(HTML(html))
            display(HTML("<span style='color:#388e3c;'>✔️ Haz clic en el botón para guardar el archivo en tu PC.</span>"))
        except Exception as e:
            display(HTML(f"<span style='color:#b71c1c;'>❌ Error al preparar la descarga: {str(e)}</span>"))

# --- Conectar eventos (solo una vez) ---
upload_widget.observe(cargar_archivos, names='value')
boton_ver.on_click(ver_datos)
boton_descargar.on_click(descargar_datos)

# --- Mostrar interfaz visual limpia y separada ---
display(HTML("""
<div style='margin-bottom:15px;'>
    <h2 style='color:#1976d2;margin:0 0 4px 0;'>📈 Sistema de gestión de datos de instrumentos</h2>
    <span style='color:#555;'>Carga, visualización y descarga de archivos CSV/XLSX</span>
</div>
"""))
display(HTML("<b>1. Subí tus archivos CSV/XLSX:</b>"))
display(upload_widget)
display(output_carga)
display(HTML("<hr style='margin:20px 0 10px 0;'>"))
display(HTML("<b>2. Seleccioná instrumento y origen de datos:</b>"))
display(widgets.HBox([instrumento_selector, origen_selector, boton_ver, boton_descargar]))
display(output_tabla)

FileUpload(value={}, accept='.csv,.xlsx', description='Subir archivos', layout=Layout(width='350px'), multiple…

Output()

Output()

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import io
from scipy import stats
from scipy.stats import normaltest, jarque_bera
from sklearn.preprocessing import StandardScaler
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Suprimir warnings para visualización limpia
warnings.filterwarnings('ignore')

# Configurar estilo de gráficos
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# =============================================================================
# MÓDULO DE ANÁLISIS EXPLORATORIO
# =============================================================================

class AnalisisExploratorio:
    def __init__(self, datos_csv, datos_xlsx):
        self.datos_csv = datos_csv
        self.datos_xlsx = datos_xlsx
        self.instrumentos = [
            "Puntos Fijos", "Piezómetros Eléctricos", "Piezómetros Casagrande",
            "Inclinómetros", "Celdas de Asentamiento", "Freatímetros", "Extensómetros"
        ]

    def obtener_datos_filtrados(self, instrumento, origen, **kwargs):
        """Obtiene los datos filtrados según los criterios del instrumento"""
        if instrumento == "Puntos Fijos":
            df_mi = self.datos_csv["puntos_fijos_mi"] if origen == "CSV" else self.datos_xlsx["puntos_fijos_mi"]
            df_md = self.datos_csv["puntos_fijos_md"] if origen == "CSV" else self.datos_xlsx["puntos_fijos_md"]
            datasets = {"Margen Izquierda (MI)": df_mi.copy(), "Margen Derecha (MD)": df_md.copy()}
            margen = kwargs.get('margen', list(datasets.keys())[0])
            df = datasets[margen]
            if kwargs.get('punto') and kwargs['punto'] != "Todos":
                df = df[df['INSTRUMENTO'] == kwargs['punto']].copy()
            if kwargs.get('anio') and kwargs['anio'] != "Todos":
                df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
                df = df[df['FECHA'].dt.year == int(kwargs['anio'])].copy()

        elif instrumento == "Piezómetros Eléctricos":
            df = (self.datos_csv["piezometros_electricos"] if origen == "CSV" else self.datos_xlsx["piezometros_electricos"]).copy()
            if kwargs.get('progresiva'):
                df = df[df['PROGRESIVA'] == kwargs['progresiva']].copy()
            if kwargs.get('piezometro') and kwargs['piezometro'] != "Todos":
                df = df[df['PIEZOMETRO'] == kwargs['piezometro']].copy()
            if kwargs.get('anio') and kwargs['anio'] != "Todos":
                df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
                df = df[df['FECHA'].dt.year == int(kwargs['anio'])].copy()

        elif instrumento == "Piezómetros Casagrande":
            df = (self.datos_csv["piezometros_casagrande"] if origen == "CSV" else self.datos_xlsx["piezometros_casagrande"]).copy()
            if kwargs.get('margen'):
                df = df[df['MARGEN'] == kwargs['margen']].copy()
            if kwargs.get('piezometro') and kwargs['piezometro'] != "Todos":
                df = df[df['PIEZOMETRO'] == kwargs['piezometro']].copy()
            if kwargs.get('anio') and kwargs['anio'] != "Todos":
                df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
                df = df[df['FECHA'].dt.year == int(kwargs['anio'])].copy()

        elif instrumento == "Inclinómetros":
            df = (self.datos_csv["inclinometros"] if origen == "CSV" else self.datos_xlsx["inclinometros"]).copy()
            if kwargs.get('inclinometro'):
                df = df[df['Inclinometro'] == kwargs['inclinometro']].copy()
            if kwargs.get('anio') and kwargs['anio'] != "Todos":
                df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
                df = df[df['Fecha'].dt.year == int(kwargs['anio'])].copy()
            if kwargs.get('eje'):
                df = df[['Fecha', 'Profundidad', kwargs['eje']]].copy()

        elif instrumento == "Celdas de Asentamiento":
            df = (self.datos_csv["asentamiento"] if origen == "CSV" else self.datos_xlsx["asentamiento"]).copy()
            if kwargs.get('progresiva'):
                df = df[df['PROGRESIVA'] == kwargs['progresiva']].copy()
            if kwargs.get('celda') and kwargs['celda'] != "Todas":
                df = df[df['CELDA_DE_ASENTAMIENTO'] == kwargs['celda']].copy()
            if kwargs.get('anio') and kwargs['anio'] != "Todos":
                df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
                df = df[df['FECHA'].dt.year == int(kwargs['anio'])].copy()

        elif instrumento == "Freatímetros":
            df = (self.datos_csv["freatimetros"] if origen == "CSV" else self.datos_xlsx["freatimetros"]).copy()
            if kwargs.get('freatimetro') and kwargs['freatimetro'] != "Todos":
                df = df[df['FREATIMETRO'] == kwargs['freatimetro']].copy()
            if kwargs.get('anio') and kwargs['anio'] != "Todos":
                df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
                df = df[df['FECHA'].dt.year == int(kwargs['anio'])].copy()

        elif instrumento == "Extensómetros":
            df = (self.datos_csv["extensometro"] if origen == "CSV" else self.datos_xlsx["extensometro"]).copy()
            if kwargs.get('extensometro') and kwargs['extensometro'] != "Todos":
                df = df[df['EXTENSOMETRO'] == kwargs['extensometro']].copy()
            if kwargs.get('anio') and kwargs['anio'] != "Todos":
                df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
                df = df[df['FECHA'].dt.year == int(kwargs['anio'])].copy()

        return df.copy()

    def detectar_columnas_temporales(self, df):
        """Detecta automáticamente columnas que pueden ser fechas"""
        columnas_temporales = []
        for col in df.columns:
            if df[col].dtype == 'object':
                try:
                    muestra = df[col].dropna().iloc[:5]
                    pd.to_datetime(muestra)
                    columnas_temporales.append(col)
                except:
                    continue
        return columnas_temporales

    def detectar_columnas_numericas(self, df):
        """Detecta columnas numéricas excluyendo las temporales"""
        numericas = df.select_dtypes(include=[np.number]).columns.tolist()
        return numericas

    def estadisticas_basicas(self, instrumento, origen, **kwargs):
        """Genera estadísticas descriptivas básicas"""
        df = self.obtener_datos_filtrados(instrumento, origen, **kwargs)

        if df.empty:
            return "No hay datos disponibles"

        info = {
            'total_registros': len(df),
            'total_columnas': len(df.columns),
            'columnas': df.columns.tolist(),
            'tipos_datos': df.dtypes.to_dict(),
            'valores_nulos': df.isnull().sum().to_dict(),
            'duplicados': df.duplicated().sum()
        }

        numericas = self.detectar_columnas_numericas(df)
        if numericas:
            info['estadisticas_numericas'] = df[numericas].describe().to_dict()

        return info

    def analisis_valores_faltantes(self, instrumento, origen, **kwargs):
        """Analiza patrones de valores faltantes"""
        df = self.obtener_datos_filtrados(instrumento, origen, **kwargs)

        if df.empty:
            return None

        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        sns.heatmap(df.isnull(), cbar=True, ax=axes[0], cmap='viridis')
        axes[0].set_title('Mapa de Valores Faltantes')
        axes[0].set_xlabel('Columnas')
        axes[0].set_ylabel('Registros')

        missing_pct = (df.isnull().sum() / len(df)) * 100
        missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

        if not missing_pct.empty:
            axes[1].bar(range(len(missing_pct)), missing_pct.values)
            axes[1].set_xticks(range(len(missing_pct)))
            axes[1].set_xticklabels(missing_pct.index, rotation=45)
            axes[1].set_title('Porcentaje de Valores Faltantes por Columna')
            axes[1].set_ylabel('Porcentaje (%)')
        else:
            axes[1].text(0.5, 0.5, 'No hay valores faltantes',
                        ha='center', va='center', transform=axes[1].transAxes)
            axes[1].set_title('Sin Valores Faltantes')

        plt.tight_layout()
        plt.show()

        return missing_pct

    def analisis_outliers(self, instrumento, origen, **kwargs):
        """Detecta y visualiza outliers usando IQR y Z-score"""
        df = self.obtener_datos_filtrados(instrumento, origen, **kwargs)

        if df.empty:
            return None

        numericas = self.detectar_columnas_numericas(df)

        if not numericas:
            print("No se encontraron columnas numéricas")
            return None

        outliers_info = {}
        fig, axes = plt.subplots(2, len(numericas), figsize=(4*len(numericas), 10))
        if len(numericas) == 1:
            axes = axes.reshape(2, 1)

        for i, col in enumerate(numericas):
            data = df[col].dropna()
            Q1 = data.quantile(0.25)
            Q3 = data.quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            outliers_iqr = data[(data < lower_bound) | (data > upper_bound)]
            z_scores = np.abs(stats.zscore(data))
            outliers_zscore = data[z_scores > 3]

            outliers_info[col] = {
                'outliers_iqr': len(outliers_iqr),
                'outliers_zscore': len(outliers_zscore),
                'porcentaje_iqr': (len(outliers_iqr) / len(data)) * 100,
                'porcentaje_zscore': (len(outliers_zscore) / len(data)) * 100
            }

            axes[0, i].boxplot(data)
            axes[0, i].set_title(f'Boxplot - {col}')
            axes[0, i].set_ylabel('Valores')

            axes[1, i].hist(data, bins=30, alpha=0.7, density=True)
            axes[1, i].set_title(f'Distribución - {col}')
            axes[1, i].set_xlabel('Valores')
            axes[1, i].set_ylabel('Densidad')

        plt.tight_layout()
        plt.show()

        return outliers_info

    def analisis_correlacion(self, instrumento, origen, **kwargs):
        """Analiza correlaciones entre variables numéricas"""
        df = self.obtener_datos_filtrados(instrumento, origen, **kwargs)

        if df.empty:
            return None

        numericas = self.detectar_columnas_numericas(df)

        if len(numericas) < 2:
            print("Se necesitan al menos 2 columnas numéricas para el análisis de correlación")
            return None

        corr_matrix = df[numericas].corr()

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
                   square=True, ax=axes[0])
        axes[0].set_title('Matriz de Correlación')

        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm',
                   center=0, square=True, ax=axes[1])
        axes[1].set_title('Matriz de Correlación (Triangular)')

        plt.tight_layout()
        plt.show()

        correlaciones_altas = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                corr_val = corr_matrix.iloc[i, j]
                if abs(corr_val) > 0.7:
                    correlaciones_altas.append({
                        'variable_1': corr_matrix.columns[i],
                        'variable_2': corr_matrix.columns[j],
                        'correlacion': corr_val
                    })

        return correlaciones_altas

    def analisis_temporal(self, instrumento, origen, columna_fecha=None, **kwargs):
        """Analiza patrones temporales en los datos"""
        df = self.obtener_datos_filtrados(instrumento, origen, **kwargs)

        if df.empty:
            return None

        if columna_fecha is None:
            columnas_temporales = self.detectar_columnas_temporales(df)
            if not columnas_temporales:
                print("No se encontraron columnas temporales")
                return None
            columna_fecha = columnas_temporales[0]

        try:
            df[columna_fecha] = pd.to_datetime(df[columna_fecha], dayfirst=True, errors='coerce')
        except:
            print(f"Error al convertir {columna_fecha} a fecha")
            return None

        df = df.sort_values(columna_fecha)
        numericas = self.detectar_columnas_numericas(df)

        if not numericas:
            print("No se encontraron columnas numéricas para análisis temporal")
            return None

        fig, axes = plt.subplots(len(numericas), 1, figsize=(15, 4*len(numericas)))
        if len(numericas) == 1:
            axes = [axes]

        for i, col in enumerate(numericas):
            axes[i].plot(df[columna_fecha], df[col], linewidth=1)
            axes[i].set_title(f'Serie Temporal - {col}')
            axes[i].set_xlabel('Fecha')
            axes[i].set_ylabel(col)
            axes[i].grid(True, alpha=0.3)

            if len(df) > 1:
                z = np.polyfit(range(len(df)), df[col].fillna(df[col].mean()), 1)
                p = np.poly1d(z)
                axes[i].plot(df[columna_fecha], p(range(len(df))),
                           "r--", alpha=0.8, label='Tendencia')
                axes[i].legend()

        plt.tight_layout()
        plt.show()

        stats_temporales = {
            'rango_fechas': (df[columna_fecha].min(), df[columna_fecha].max()),
            'total_dias': (df[columna_fecha].max() - df[columna_fecha].min()).days,
            'frecuencia_promedio': len(df) / max(1, (df[columna_fecha].max() - df[columna_fecha].min()).days)
        }

        return stats_temporales

    def generar_reporte_completo(self, instrumento, origen, **kwargs):
        """Genera un reporte completo del instrumento"""
        print(f"🔍 ANÁLISIS EXPLORATORIO - {instrumento.upper()} ({origen.upper()})")
        print("=" * 70)

        print("\n📊 ESTADÍSTICAS BÁSICAS:")
        stats = self.estadisticas_basicas(instrumento, origen, **kwargs)
        if isinstance(stats, dict):
            print(f"  • Total registros: {stats['total_registros']:,}")
            print(f"  • Total columnas: {stats['total_columnas']}")
            print(f"  • Registros duplicados: {stats['duplicados']}")
            print(f"  • Columnas: {', '.join(stats['columnas'])}")

        print("\n🔍 ANÁLISIS DE VALORES FALTANTES:")
        missing = self.analisis_valores_faltantes(instrumento, origen, **kwargs)

        print("\n⚠️ ANÁLISIS DE OUTLIERS:")
        outliers = self.analisis_outliers(instrumento, origen, **kwargs)
        if outliers:
            for col, info in outliers.items():
                print(f"  • {col}: {info['outliers_iqr']} outliers IQR ({info['porcentaje_iqr']:.1f}%)")

        print("\n🔗 ANÁLISIS DE CORRELACIÓN:")
        correlaciones = self.analisis_correlacion(instrumento, origen, **kwargs)
        if correlaciones:
            for corr in correlaciones:
                print(f"  • {corr['variable_1']} ↔ {corr['variable_2']}: {corr['correlacion']:.3f}")

        print("\n⏱️ ANÁLISIS TEMPORAL:")
        temporal = self.analisis_temporal(instrumento, origen, **kwargs)
        if temporal:
            print(f"  • Rango: {temporal['rango_fechas'][0]} a {temporal['rango_fechas'][1]}")
            print(f"  • Duración: {temporal['total_dias']} días")
            print(f"  • Frecuencia promedio: {temporal['frecuencia_promedio']:.2f} registros/día")

# =============================================================================
# INTERFAZ GRÁFICA PARA EDA
# =============================================================================

# Crear instancia del analizador
analizador = AnalisisExploratorio(datos_csv, datos_xlsx)

# Selectores comunes
instrumento_dropdown = widgets.Dropdown(
    options=["Puntos Fijos", "Piezómetros Eléctricos", "Piezómetros Casagrande", "Inclinómetros",
             "Celdas de Asentamiento", "Freatímetros", "Extensómetros"],
    value="Puntos Fijos",
    description="Instrumento:"
)
origen_dropdown = widgets.Dropdown(
    options=["CSV", "XLSX"],
    value="CSV",
    description="Origen:"
)

# Selectores por instrumento
margen_dropdown = widgets.Dropdown(description="Margen:")
punto_dropdown = widgets.Dropdown(description="Punto Fijo:")
variable_pf_dropdown = widgets.Dropdown(description="Variable:")
anio_pf_dropdown = widgets.Dropdown(description="Año:")

progresiva_dropdown = widgets.Dropdown(description="Progresiva:")
piezometro_dropdown = widgets.Dropdown(description="Piezómetro:")
variable_pe_dropdown = widgets.Dropdown(description="Variable:")
anio_pe_dropdown = widgets.Dropdown(description="Año:")

margen_cg_dropdown = widgets.Dropdown(description="Margen:")
pz_cg_dropdown = widgets.Dropdown(description="Piezómetro:")
variable_cg_dropdown = widgets.Dropdown(description="Variable:")
anio_cg_dropdown = widgets.Dropdown(description="Año:")

inclinometro_dropdown = widgets.Dropdown(description="Inclinómetro:")
anio_inc_dropdown = widgets.Dropdown(description="Año:")
eje_dropdown = widgets.Dropdown(
    options=["A+", "A-", "B+", "B-"],
    value="A+",
    description="Eje:"
)

progresiva_ca_dropdown = widgets.Dropdown(description="Progresiva:")
celda_dropdown = widgets.Dropdown(description="Celda:")
variable_ca_dropdown = widgets.Dropdown(description="Variable:")
anio_ca_dropdown = widgets.Dropdown(description="Año:")

freatimetro_dropdown = widgets.Dropdown(description="Freatímetro:")
variable_fr_dropdown = widgets.Dropdown(description="Variable:")
anio_fr_dropdown = widgets.Dropdown(description="Año:")

extensometro_dropdown = widgets.Dropdown(description="Extensómetro:")
variable_ex_dropdown = widgets.Dropdown(description="Variable:")
anio_ex_dropdown = widgets.Dropdown(description="Año:")

# Selector de tipo de análisis
eda_analisis_selector = widgets.Dropdown(
    options=[
        ('Reporte Completo', 'completo'),
        ('Estadísticas Básicas', 'basicas'),
        ('Valores Faltantes', 'faltantes'),
        ('Outliers', 'outliers'),
        ('Correlación', 'correlacion'),
        ('Análisis Temporal', 'temporal')
    ],
    description='Análisis:',
    layout=widgets.Layout(width='200px')
)

# Botones y salida
boton_analizar = widgets.Button(
    description='📊 Analizar',
    button_style='primary',
    icon='chart-line',
    layout=widgets.Layout(width='120px')
)
output_eda = widgets.Output()

# Funciones para actualizar opciones
def actualizar_opciones_pf(change=None):
    origen = origen_dropdown.value
    df_mi = (datos_csv["puntos_fijos_mi"] if origen == "CSV" else datos_xlsx["puntos_fijos_mi"]).copy()
    df_md = (datos_csv["puntos_fijos_md"] if origen == "CSV" else datos_xlsx["puntos_fijos_md"]).copy()
    datasets = {"Margen Izquierda (MI)": df_mi, "Margen Derecha (MD)": df_md}
    datasets = {k: v for k, v in datasets.items() if not v.empty}
    if not datasets:
        margen_dropdown.options = []
        punto_dropdown.options = []
        variable_pf_dropdown.options = []
        anio_pf_dropdown.options = []
        return
    margen_dropdown.options = list(datasets.keys())
    margen = margen_dropdown.value or list(datasets.keys())[0]
    df = datasets[margen].copy()
    df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
    variable_pf_dropdown.options = [col for col in df.select_dtypes(include='number').columns if col not in ['FECHA', 'INSTRUMENTO', 'MARGEN']]
    punto_dropdown.options = ["Todos"] + sorted(df['INSTRUMENTO'].dropna().unique())
    anio_pf_dropdown.options = ["Todos"] + [str(y) for y in sorted(df['FECHA'].dt.year.dropna().unique())]

def actualizar_opciones_pe(change=None):
    df = (datos_csv["piezometros_electricos"] if origen_dropdown.value == "CSV" else datos_xlsx["piezometros_electricos"]).copy()
    if df.empty or 'PROGRESIVA' not in df.columns:
        progresiva_dropdown.options = []
        piezometro_dropdown.options = []
        variable_pe_dropdown.options = []
        anio_pe_dropdown.options = []
        return
    df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
    progresiva_dropdown.options = sorted(df['PROGRESIVA'].dropna().unique())
    if progresiva_dropdown.options:
        progresiva_dropdown.value = progresiva_dropdown.options[0]
    actualizar_piezometros_pe()
    columnas_excluir = ['FECHA', 'PROGRESIVA', 'PIEZOMETRO']
    variables = [c for c in df.columns if c not in columnas_excluir]
    variable_pe_dropdown.options = variables
    if variables:
        variable_pe_dropdown.value = variables[0]
    anio_pe_dropdown.options = ["Todos"] + [str(y) for y in sorted(df['FECHA'].dt.year.dropna().unique())]

def actualizar_piezometros_pe(change=None):
    df = (datos_csv["piezometros_electricos"] if origen_dropdown.value == "CSV" else datos_xlsx["piezometros_electricos"]).copy()
    if df.empty or 'PROGRESIVA' not in df.columns or 'PIEZOMETRO' not in df.columns:
        piezometro_dropdown.options = []
        return
    piezos = sorted(df[df['PROGRESIVA'] == progresiva_dropdown.value]['PIEZOMETRO'].dropna().unique())
    piezometro_dropdown.options = ["Todos"] + list(piezos)
    if piezometro_dropdown.options:
        piezometro_dropdown.value = piezometro_dropdown.options[0]

def actualizar_opciones_cg(change=None):
    df = (datos_csv["piezometros_casagrande"] if origen_dropdown.value == "CSV" else datos_xlsx["piezometros_casagrande"]).copy()
    if df.empty or 'MARGEN' not in df.columns:
        margen_cg_dropdown.options = []
        pz_cg_dropdown.options = []
        variable_cg_dropdown.options = []
        anio_cg_dropdown.options = []
        return
    df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
    margen_cg_dropdown.options = sorted(df['MARGEN'].dropna().unique())
    if margen_cg_dropdown.options:
        margen_cg_dropdown.value = margen_cg_dropdown.options[0]
    actualizar_piezometros_cg()
    columnas_excluir = ['FECHA', 'MARGEN', 'PIEZOMETRO']
    variables = [c for c in df.columns if c not in columnas_excluir]
    variable_cg_dropdown.options = variables
    if variables:
        variable_cg_dropdown.value = variables[0]
    anio_cg_dropdown.options = ["Todos"] + [str(y) for y in sorted(df['FECHA'].dt.year.dropna().unique())]

def actualizar_piezometros_cg(change=None):
    df = (datos_csv["piezometros_casagrande"] if origen_dropdown.value == "CSV" else datos_xlsx["piezometros_casagrande"]).copy()
    if df.empty or 'MARGEN' not in df.columns or 'PIEZOMETRO' not in df.columns:
        pz_cg_dropdown.options = []
        return
    piezos = sorted(df[df['MARGEN'] == margen_cg_dropdown.value]['PIEZOMETRO'].dropna().unique())
    pz_cg_dropdown.options = ["Todos"] + list(piezos)
    if pz_cg_dropdown.options:
        pz_cg_dropdown.value = pz_cg_dropdown.options[0]

def actualizar_opciones_inc(change=None):
    df = (datos_csv["inclinometros"] if origen_dropdown.value == "CSV" else datos_xlsx["inclinometros"]).copy()
    if df.empty:
        inclinometro_dropdown.options = []
        anio_inc_dropdown.options = []
        return
    df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
    inclinometro_dropdown.options = sorted(df['Inclinometro'].dropna().unique())
    anio_inc_dropdown.options = ["Todos"] + [str(y) for y in sorted(df['Fecha'].dt.year.dropna().unique())]

def actualizar_opciones_ca(change=None):
    df = (datos_csv["asentamiento"] if origen_dropdown.value == "CSV" else datos_xlsx["asentamiento"]).copy()
    if df.empty:
        progresiva_ca_dropdown.options = []
        celda_dropdown.options = []
        variable_ca_dropdown.options = []
        anio_ca_dropdown.options = []
        return
    df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
    progresiva_ca_dropdown.options = sorted(df['PROGRESIVA'].dropna().unique())
    actualizar_celdas_ca()
    variable_ca_dropdown.options = [c for c in df.select_dtypes(include='number').columns if c not in ['FECHA', 'PROGRESIVA', 'CELDA_DE_ASENTAMIENTO']]
    anio_ca_dropdown.options = ["Todos"] + [str(y) for y in sorted(df['FECHA'].dt.year.dropna().unique())]

def actualizar_celdas_ca(change=None):
    df = (datos_csv["asentamiento"] if origen_dropdown.value == "CSV" else datos_xlsx["asentamiento"]).copy()
    if df.empty or 'PROGRESIVA' not in df.columns or 'CELDA_DE_ASENTAMIENTO' not in df.columns:
        celda_dropdown.options = []
        return
    celdas = sorted(df[df['PROGRESIVA'] == progresiva_ca_dropdown.value]['CELDA_DE_ASENTAMIENTO'].dropna().unique())
    celda_dropdown.options = ["Todas"] + list(celdas)
    if celda_dropdown.options:
        celda_dropdown.value = celda_dropdown.options[0]

def actualizar_opciones_fr(change=None):
    df = (datos_csv["freatimetros"] if origen_dropdown.value == "CSV" else datos_xlsx["freatimetros"]).copy()
    if df.empty:
        freatimetro_dropdown.options = []
        variable_fr_dropdown.options = []
        anio_fr_dropdown.options = []
        return
    df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
    freatimetro_dropdown.options = ["Todos"] + sorted(df['FREATIMETRO'].dropna().unique())
    variable_fr_dropdown.options = [c for c in df.select_dtypes(include='number').columns if c not in ['FECHA', 'FREATIMETRO']]
    anio_fr_dropdown.options = ["Todos"] + [str(y) for y in sorted(df['FECHA'].dt.year.dropna().unique())]

def actualizar_opciones_ex(change=None):
    df = (datos_csv["extensometro"] if origen_dropdown.value == "CSV" else datos_xlsx["extensometro"]).copy()
    if df.empty:
        extensometro_dropdown.options = []
        variable_ex_dropdown.options = []
        anio_ex_dropdown.options = []
        return
    df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
    extensometro_dropdown.options = ["Todos"] + sorted(df['EXTENSOMETRO'].dropna().unique())
    variable_ex_dropdown.options = [c for c in df.select_dtypes(include='number').columns if c not in ['FECHA', 'EXTENSOMETRO']]
    anio_ex_dropdown.options = ["Todos"] + [str(y) for y in sorted(df['FECHA'].dt.year.dropna().unique())]

def actualizar_controles_visibles(change=None):
    tipo = instrumento_dropdown.value
    for w in [margen_dropdown, punto_dropdown, variable_pf_dropdown, anio_pf_dropdown,
              progresiva_dropdown, piezometro_dropdown, variable_pe_dropdown, anio_pe_dropdown,
              margen_cg_dropdown, pz_cg_dropdown, variable_cg_dropdown, anio_cg_dropdown,
              inclinometro_dropdown, anio_inc_dropdown, eje_dropdown,
              progresiva_ca_dropdown, celda_dropdown, variable_ca_dropdown, anio_ca_dropdown,
              freatimetro_dropdown, variable_fr_dropdown, anio_fr_dropdown,
              extensometro_dropdown, variable_ex_dropdown, anio_ex_dropdown]:
        w.layout.display = 'none'

    if tipo == "Puntos Fijos":
        margen_dropdown.layout.display = 'flex'
        punto_dropdown.layout.display = 'flex'
        variable_pf_dropdown.layout.display = 'flex'
        anio_pf_dropdown.layout.display = 'flex'
        actualizar_opciones_pf()
    elif tipo == "Piezómetros Eléctricos":
        progresiva_dropdown.layout.display = 'flex'
        piezometro_dropdown.layout.display = 'flex'
        variable_pe_dropdown.layout.display = 'flex'
        anio_pe_dropdown.layout.display = 'flex'
        actualizar_opciones_pe()
    elif tipo == "Piezómetros Casagrande":
        margen_cg_dropdown.layout.display = 'flex'
        pz_cg_dropdown.layout.display = 'flex'
        variable_cg_dropdown.layout.display = 'flex'
        anio_cg_dropdown.layout.display = 'flex'
        actualizar_opciones_cg()
    elif tipo == "Inclinómetros":
        inclinometro_dropdown.layout.display = 'flex'
        anio_inc_dropdown.layout.display = 'flex'
        eje_dropdown.layout.display = 'flex'
        actualizar_opciones_inc()
    elif tipo == "Celdas de Asentamiento":
        progresiva_ca_dropdown.layout.display = 'flex'
        celda_dropdown.layout.display = 'flex'
        variable_ca_dropdown.layout.display = 'flex'
        anio_ca_dropdown.layout.display = 'flex'
        actualizar_opciones_ca()
    elif tipo == "Freatímetros":
        freatimetro_dropdown.layout.display = 'flex'
        variable_fr_dropdown.layout.display = 'flex'
        anio_fr_dropdown.layout.display = 'flex'
        actualizar_opciones_fr()
    elif tipo == "Extensómetros":
        extensometro_dropdown.layout.display = 'flex'
        variable_ex_dropdown.layout.display = 'flex'
        anio_ex_dropdown.layout.display = 'flex'
        actualizar_opciones_ex()

def ejecutar_analisis(b):
    with output_eda:
        clear_output(wait=True)
        instrumento = instrumento_dropdown.value
        origen = origen_dropdown.value
        tipo_analisis = eda_analisis_selector.value

        kwargs = {}
        if instrumento == "Puntos Fijos":
            kwargs = {'margen': margen_dropdown.value, 'punto': punto_dropdown.value, 'anio': anio_pf_dropdown.value}
        elif instrumento == "Piezómetros Eléctricos":
            kwargs = {'progresiva': progresiva_dropdown.value, 'piezometro': piezometro_dropdown.value, 'anio': anio_pe_dropdown.value}
        elif instrumento == "Piezómetros Casagrande":
            kwargs = {'margen': margen_cg_dropdown.value, 'piezometro': pz_cg_dropdown.value, 'anio': anio_cg_dropdown.value}
        elif instrumento == "Inclinómetros":
            kwargs = {'inclinometro': inclinometro_dropdown.value, 'anio': anio_inc_dropdown.value, 'eje': eje_dropdown.value}
        elif instrumento == "Celdas de Asentamiento":
            kwargs = {'progresiva': progresiva_ca_dropdown.value, 'celda': celda_dropdown.value, 'anio': anio_ca_dropdown.value}
        elif instrumento == "Freatímetros":
            kwargs = {'freatimetro': freatimetro_dropdown.value, 'anio': anio_fr_dropdown.value}
        elif instrumento == "Extensómetros":
            kwargs = {'extensometro': extensometro_dropdown.value, 'anio': anio_ex_dropdown.value}

        df = analizador.obtener_datos_filtrados(instrumento, origen, **kwargs)
        if df.empty:
            display(HTML("<div style='color:#b71c1c;font-weight:bold;'>⚠️ No hay datos disponibles para el análisis.</div>"))
            return

        try:
            if tipo_analisis == 'completo':
                analizador.generar_reporte_completo(instrumento, origen, **kwargs)
            elif tipo_analisis == 'basicas':
                stats = analizador.estadisticas_basicas(instrumento, origen, **kwargs)
                print("📊 ESTADÍSTICAS BÁSICAS:")
                print(f"Total registros: {stats['total_registros']:,}")
                print(f"Total columnas: {stats['total_columnas']}")
                print(f"Columnas: {', '.join(stats['columnas'])}")
                if 'estadisticas_numericas' in stats:
                    df_stats = pd.DataFrame(stats['estadisticas_numericas'])
                    display(df_stats.round(3))
            elif tipo_analisis == 'faltantes':
                analizador.analisis_valores_faltantes(instrumento, origen, **kwargs)
            elif tipo_analisis == 'outliers':
                analizador.analisis_outliers(instrumento, origen, **kwargs)
            elif tipo_analisis == 'correlacion':
                analizador.analisis_correlacion(instrumento, origen, **kwargs)
            elif tipo_analisis == 'temporal':
                analizador.analisis_temporal(instrumento, origen, **kwargs)
        except Exception as e:
            display(HTML(f"<div style='color:#b71c1c;'>❌ Error en el análisis: {str(e)}</div>"))

# Conectar eventos
instrumento_dropdown.observe(actualizar_controles_visibles, names='value')
origen_dropdown.observe(actualizar_controles_visibles, names='value')
margen_dropdown.observe(actualizar_opciones_pf, names='value')
progresiva_dropdown.observe(actualizar_piezometros_pe, names='value')
margen_cg_dropdown.observe(actualizar_piezometros_cg, names='value')
progresiva_ca_dropdown.observe(actualizar_celdas_ca, names='value')
boton_analizar.on_click(ejecutar_analisis)

# Mostrar interfaz
display(HTML("<hr style='margin:20px 0 15px 0;'>"))
display(HTML("""
<div style='margin-bottom:15px;'>
    <h2 style='color:#1976d2;margin:0 0 4px 0;'>🔍 Análisis Exploratorio de Datos (EDA)</h2>
    <span style='color:#555;'>Análisis estadístico profundo de los datos de instrumentos</span>
</div>
"""))
display(HTML("<b>1. Selecciona el instrumento y origen:</b>"))
display(widgets.HBox([instrumento_dropdown, origen_dropdown]))
display(HTML("<b>2. Selecciona los parámetros específicos:</b>"))
display(widgets.HBox([
    margen_dropdown, punto_dropdown, variable_pf_dropdown, anio_pf_dropdown,
    progresiva_dropdown, piezometro_dropdown, variable_pe_dropdown, anio_pe_dropdown,
    margen_cg_dropdown, pz_cg_dropdown, variable_cg_dropdown, anio_cg_dropdown,
    inclinometro_dropdown, anio_inc_dropdown, eje_dropdown,
    progresiva_ca_dropdown, celda_dropdown, variable_ca_dropdown, anio_ca_dropdown,
    freatimetro_dropdown, variable_fr_dropdown, anio_fr_dropdown,
    extensometro_dropdown, variable_ex_dropdown, anio_ex_dropdown
]))
display(HTML("<b>3. Selecciona el tipo de análisis:</b>"))
display(widgets.HBox([eda_analisis_selector, boton_analizar]))
display(output_eda)

# Inicializar
actualizar_controles_visibles()

Output()